# Prototype-Contrastive Explainable Statute Prediction (InLegalBERT)

Pipeline (matches Fig. 1 of the paper):

1. **PySBD** splits each case into factual sentences.
2. Every IPC section description is a **statute prototype** (511 prototypes = the *Statute Prototype Bank*).
3. One **shared InLegalBERT** encoder embeds case sentences and statute prototypes.
   * **Run 1** – *prototype-contrastive* fine-tuning (positive prototype + mined hard-negative prototypes, InfoNCE, τ = 0.05, 4 hard negatives).
   * **Run 2** – original InLegalBERT, no fine-tuning (baseline).
4. **Sentence–Prototype Similarity Matrix** (cosine) → **Prototype Ranking** (max sentence similarity).
5. **Evidence selection**: top-scoring sentences for each predicted prototype.
6. **Explanation**: Qwen LLM (optional) or a template fallback, grounded in the prototype text and the evidence.

Both runs are executed and compared at the end of the notebook.

## 1. Setup

In [ ]:
import subprocess, sys, importlib

def ensure_packages():
    pkgs = {"torch": "torch", "transformers": "transformers", "scikit-learn": "sklearn",
            "nltk": "nltk", "numpy": "numpy", "pandas": "pandas",
            "scikit-multilearn": "skmultilearn", "tqdm": "tqdm", "pysbd": "pysbd"}
    for pip_name, import_name in pkgs.items():
        try:
            importlib.import_module(import_name)
        except ImportError:
            print(f"[setup] installing {pip_name} ...")
            subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", pip_name], check=True)

ensure_packages()

import os, re, json, csv, random, difflib
from collections import Counter
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

import pysbd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (f1_score, precision_score, recall_score, accuracy_score,
                             fbeta_score, hamming_loss, classification_report)
from transformers import AutoTokenizer, AutoModel

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print("Device:", DEVICE, "| AMP:", USE_AMP)

## 2. Configuration

In [ ]:
@dataclass
class ProtoConfig:
    # data
    train_path: str = "Pavithra/TAsk1/task1.jsonl"                       # {doc_id, fact, statute, explanation}
    prototype_source_path: str = "Pavithra/TAsk1/ipc_sections_clean.json"  # 511 IPC section descriptions

    # shared encoder
    encoder_name: str = "law-ai/InLegalBERT"
    freeze_layers: int = 0            # raise (e.g. 6) if GPU memory is tight
    max_length: int = 384             # sentence and prototype max token length
    max_sentences: int = 60

    # prototype-contrastive training (Run 1)
    temperature: float = 0.05
    num_hard_negatives: int = 4       # hard-negative prototypes per positive prototype
    exclude_same_base_section: bool = True   # never use IPC 498 as a negative for 498A
    encoder_lr: float = 2e-5
    weight_decay: float = 0.01
    batch_size: int = 8
    grad_accum_steps: int = 2
    num_epochs: int = 8
    grad_clip: float = 1.0

    # split
    train_fraction: float = 0.70
    val_fraction: float = 0.10
    test_fraction: float = 0.20

    # prototype ranking -> decision rule
    evidence_per_prototype: int = 2         # evidence sentences kept per predicted prototype
    use_calibrated_thresholds: bool = True  # per-prototype thresholds tuned on validation
    threshold_grid: tuple = tuple(round(x, 2) for x in np.arange(0.10, 0.91, 0.02))
    calibration_fbeta: float = 0.7
    default_threshold: float = 0.55
    second_label_margin: float = 0.08
    top_k_fallback: int = 1

    # explanation generator
    use_qwen_explainer: bool = False
    qwen_model_name: str = "Qwen/Qwen2.5-1.5B-Instruct"
    qwen_max_new_tokens: int = 160

    out_dir: str = "./prototype_contrastive_outputs"

cfg = ProtoConfig()
os.makedirs(cfg.out_dir, exist_ok=True)
cfg

## 3. Case data and Statute Prototype Bank

In [ ]:
def load_jsonl_dataset(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            stat = rec.get("statute", [])
            if isinstance(stat, str):
                stat = [stat]
            rec["statute"] = [str(s).strip() for s in stat]
            records.append(rec)
    return records


def normalize_ipc_label(label):
    m = re.search(r"(\d+[A-Za-z]*)", str(label).strip())
    return f"IPC {m.group(1).upper()}" if m else None


def base_number_of(section_code):
    m = re.search(r"(\d+)", section_code)
    return m.group(1) if m else section_code


def load_statute_prototypes(path):
    """Returns (prototype_texts {code: description}, prototype_titles {code: title})."""
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)
    texts, titles = {}, {}

    def store(code_raw, text, title=None):
        code = normalize_ipc_label(code_raw)
        if code and text and str(text).strip():
            texts[code] = str(text).strip()
            if title:
                titles[code] = str(title).strip()

    def pick_text(d):
        return (d.get("description") or d.get("text") or d.get("content") or d.get("definition")
                or d.get("summary") or d.get("section_desc") or "")

    def pick_title(d):
        return d.get("title") or d.get("name") or d.get("offense") or d.get("heading")

    if isinstance(raw, dict):
        for k, v in raw.items():
            if isinstance(v, str):
                store(k, v)
            elif isinstance(v, dict):
                store(k, pick_text(v), pick_title(v))
    elif isinstance(raw, list):
        for item in raw:
            if not isinstance(item, dict):
                continue
            code_raw = (item.get("section") or item.get("section_number") or item.get("code")
                        or item.get("ipc_section") or item.get("id") or item.get("section_no"))
            if code_raw is not None:
                store(code_raw, pick_text(item), pick_title(item))
    else:
        raise ValueError(f"Unrecognised prototype file schema: {type(raw)}")
    return texts, titles


records = load_jsonl_dataset(cfg.train_path)
prototype_texts, prototype_titles = load_statute_prototypes(cfg.prototype_source_path)
prototype_codes = sorted(prototype_texts.keys())
print(f"{len(records)} cases | {len(prototype_codes)} statute prototypes")
for c in prototype_codes[:3]:
    print(f"  {c}: {prototype_texts[c][:110]}...")
if not prototype_codes:
    print("WARNING: 0 prototypes parsed - check the field names in load_statute_prototypes().")

gold_seen = sorted({normalize_ipc_label(s) for r in records for s in r["statute"]} - {None})
missing = sorted(set(gold_seen) - set(prototype_codes))
if missing:
    print("WARNING: gold sections absent from the Prototype Bank:", missing)
print(f"{len(gold_seen)} supervised sections out of {len(prototype_codes)} prototypes")

## 4. PySBD sentence splitting and (sentence, positive prototype) pairs

The positive pair in Fig. 1 is *(case sentence, applicable statute prototype)*, built from the
per-sentence `explanation` field (exact match first, fuzzy fallback).

In [ ]:
_segmenter = pysbd.Segmenter(language="en", clean=False)

def split_sentences(text, max_sentences=None):
    sents = [s.strip() for s in _segmenter.segment(text or "") if s.strip()]
    if not sents:
        sents = [text.strip()] if text and text.strip() else ["."]
    return sents[:max_sentences] if max_sentences else sents


def positive_pairs_from_explanation(fact, explanation):
    """(case sentence, positive prototype code) pairs."""
    sentences = split_sentences(fact, cfg.max_sentences)
    pairs = []
    for exp_sent, label in (explanation or {}).items():
        code = normalize_ipc_label(label)
        if not code or code not in prototype_texts:
            continue
        exp_norm = re.sub(r"\s+", " ", exp_sent).strip()
        best_s, best_r = None, 0.0
        for s in sentences:
            r = difflib.SequenceMatcher(None, re.sub(r"\s+", " ", s).strip(), exp_norm, autojunk=False).ratio()
            if r > best_r:
                best_r, best_s = r, s
        if best_s is not None and best_r >= 0.5:
            pairs.append((best_s, code))
    return pairs


for rec in tqdm(records, desc="PySBD"):
    rec["sentences"] = split_sentences(rec["fact"], cfg.max_sentences)
    rec["gold_sections"] = sorted({s for s in (normalize_ipc_label(g) for g in rec["statute"]) if s})

label_counts = Counter(s for r in records for s in r["gold_sections"])
print(dict(sorted(label_counts.items(), key=lambda x: -x[1])))

## 5. Train / validation / test split (70 / 10 / 20, multilabel-stratified)

In [ ]:
def multilabel_stratified_split(docs, fraction, seed):
    from skmultilearn.model_selection import iterative_train_test_split
    labels = sorted({s for d in docs for s in d["gold_sections"]})
    y = MultiLabelBinarizer(classes=labels).fit_transform([d["gold_sections"] for d in docs])
    X = np.arange(len(docs)).reshape(-1, 1)
    np.random.seed(seed)
    X_keep, _, X_held, _ = iterative_train_test_split(X, y, test_size=fraction)
    keep, held = set(X_keep.flatten().tolist()), set(X_held.flatten().tolist())
    return [docs[i] for i in range(len(docs)) if i in keep], [docs[i] for i in range(len(docs)) if i in held]


remainder, test_split = multilabel_stratified_split(records, cfg.test_fraction, SEED)
train_split, val_split = multilabel_stratified_split(
    remainder, cfg.val_fraction / (cfg.train_fraction + cfg.val_fraction), SEED + 1)
print(f"Train {len(train_split)} | Val {len(val_split)} | Test {len(test_split)}")

## 6. Shared InLegalBERT prototype encoder

One encoder embeds both case sentences and statute prototypes (bi-encoder / Siamese setup).

In [ ]:
def mean_pool(hidden, mask):
    m = mask.unsqueeze(-1).float()
    return (hidden * m).sum(1) / m.sum(1).clamp(min=1e-9)


class PrototypeEncoder(nn.Module):
    """Shared InLegalBERT encoder producing L2-normalised embeddings for sentences and prototypes."""

    def __init__(self, model_name, freeze_layers=0):
        super().__init__()
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.bert = AutoModel.from_pretrained(model_name)
        self.embed_dim = self.bert.config.hidden_size
        self.freeze_bottom(freeze_layers)

    def freeze_bottom(self, n):
        if n <= 0:
            return
        for p in self.bert.embeddings.parameters():
            p.requires_grad = False
        for i, layer in enumerate(self.bert.encoder.layer):
            for p in layer.parameters():
                p.requires_grad = i >= n
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"Encoder: {trainable:,} trainable parameters (bottom {n} layers frozen)")

    def forward(self, texts, max_length):
        enc = self.tokenizer(texts, truncation=True, padding=True, max_length=max_length,
                             return_tensors="pt").to(next(self.parameters()).device)
        hidden = self.bert(**enc).last_hidden_state
        return F.normalize(mean_pool(hidden, enc["attention_mask"]), p=2, dim=-1)

    @torch.no_grad()
    def embed(self, texts, max_length, batch_size=32):
        self.eval()
        out = [self(texts[i:i + batch_size], max_length).float().cpu()
               for i in range(0, len(texts), batch_size)]
        return torch.cat(out, 0) if out else torch.zeros((0, self.embed_dim))


encoder = PrototypeEncoder(cfg.encoder_name, cfg.freeze_layers).to(DEVICE)

## 7. Statute Prototype Bank + hard-negative prototype mining

`build_prototype_bank` embeds all 511 prototypes with the current encoder.
`mine_hard_negative_prototypes` picks, for every prototype, the most similar prototypes that are
**legally different** (a different base IPC section), which serve as its hard negatives.

In [ ]:
def build_prototype_bank(enc):
    """Embeddings of all statute prototypes, shape (num_prototypes, dim)."""
    return enc.embed([prototype_texts[c] for c in prototype_codes], cfg.max_length)


def mine_hard_negative_prototypes(bank_emb, k, exclude_same_base=True):
    sim = bank_emb @ bank_emb.t()
    hard = {}
    for i, code in enumerate(prototype_codes):
        negatives = []
        for j in torch.argsort(sim[i], descending=True).tolist():
            other = prototype_codes[j]
            if j == i:
                continue
            if exclude_same_base and base_number_of(other) == base_number_of(code):
                continue
            negatives.append(other)
            if len(negatives) == k:
                break
        hard[code] = negatives
    return hard

## 8. Scoring, calibration and prediction (shared by Run 1 and Run 2)

`score_case` builds the sentence–prototype similarity matrix, ranks prototypes by max sentence similarity
and keeps the best evidence sentences for every prototype.

In [ ]:
def score_case(fact_text, bank_emb):
    """Returns (prototype_scores {code: max sim}, evidence {code: [sentences]}, sentences)."""
    sentences = split_sentences(fact_text, cfg.max_sentences)
    sent_emb = encoder.embed(sentences, cfg.max_length)
    sim_matrix = (sent_emb @ bank_emb.t()).numpy()            # (num_sentences, num_prototypes)
    best = sim_matrix.max(axis=0)
    scores = {c: float(best[j]) for j, c in enumerate(prototype_codes)}
    evidence = {}
    for j, c in enumerate(prototype_codes):
        top_idx = np.argsort(-sim_matrix[:, j])[:cfg.evidence_per_prototype]
        evidence[c] = [sentences[i] for i in top_idx]
    return scores, evidence, sentences


def calibrate_prototype_thresholds(bank_emb):
    gold = [d["gold_sections"] for d in val_split]
    val_scores = [score_case(d["fact"], bank_emb)[0] for d in tqdm(val_split, desc="Scoring val")]
    thresholds = {}
    for code in sorted({s for g in gold for s in g}):
        y_true = np.array([1.0 if code in g else 0.0 for g in gold])
        vals = np.array([sc.get(code, 0.0) for sc in val_scores])
        best_t, best_f = cfg.default_threshold, -1.0
        for t in cfg.threshold_grid:
            f = fbeta_score(y_true, (vals >= t).astype(int), beta=cfg.calibration_fbeta, zero_division=0)
            if f > best_f:
                best_f, best_t = f, float(t)
        thresholds[code] = best_t
    return thresholds


def predict_case(fact_text, bank_emb, thresholds):
    scores, evidence, _ = score_case(fact_text, bank_emb)
    if cfg.use_calibrated_thresholds:
        hits = [(c, s) for c, s in scores.items() if s >= thresholds.get(c, cfg.default_threshold)]
    else:
        hits = []
    hits.sort(key=lambda x: -x[1])
    if hits:
        top = hits[0][1]
        chosen = [hits[0]] + [h for h in hits[1:] if h[1] >= top - cfg.second_label_margin]
    else:
        ranked = sorted(scores.items(), key=lambda x: -x[1])
        chosen = ranked[:cfg.top_k_fallback]
    return [{"section": c, "score": round(float(s), 4), "evidence_sentences": evidence[c]} for c, s in chosen]

## 9. Evidence-grounded explanation

Uses a Qwen LLM when `cfg.use_qwen_explainer = True`; otherwise a template that still cites the
prototype text and the selected evidence sentences.

In [ ]:
_qwen = {}

def _load_qwen():
    if "model" not in _qwen:
        from transformers import AutoModelForCausalLM
        _qwen["tok"] = AutoTokenizer.from_pretrained(cfg.qwen_model_name)
        _qwen["model"] = AutoModelForCausalLM.from_pretrained(
            cfg.qwen_model_name, torch_dtype=torch.float16 if USE_AMP else torch.float32).to(DEVICE)
    return _qwen["tok"], _qwen["model"]


def generate_explanation(section_code, evidence_sentences, score):
    title = prototype_titles.get(section_code, "")
    prototype_text = prototype_texts.get(section_code, "")
    if cfg.use_qwen_explainer:
        tok, model = _load_qwen()
        evidence_block = "\n".join(f"- {s}" for s in evidence_sentences)
        messages = [
            {"role": "system", "content": "You are a legal assistant explaining why an IPC section applies to a case."},
            {"role": "user", "content": (f"Statute: {section_code} {title}\nStatute text: {prototype_text}\n"
                                         f"Evidence sentences from the case:\n{evidence_block}\n\n"
                                         "Write a short legal explanation linking the evidence to the statute.")},
        ]
        prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tok(prompt, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=cfg.qwen_max_new_tokens, do_sample=False)
        return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    title_part = f" ({title})" if title else ""
    ev = " | ".join(s[:140] for s in evidence_sentences)
    return (f"{section_code}{title_part} is predicted (prototype similarity {score:.3f}). "
            f"Evidence: \"{ev}\". Statute prototype: \"{prototype_text[:160]}...\". "
            f"The evidence sentences are the closest matches to this statute prototype in embedding space.")

## 10. Evaluation helper and Run 2 (original InLegalBERT prototypes)

In [ ]:
def evaluate_run(run_name, save_predictions=True):
    bank_emb = build_prototype_bank(encoder)
    thresholds = calibrate_prototype_thresholds(bank_emb) if cfg.use_calibrated_thresholds else {}

    predictions = {}
    for d in tqdm(test_split, desc=f"[{run_name}] predicting"):
        preds = predict_case(d["fact"], bank_emb, thresholds)
        for p in preds:
            p["explanation"] = generate_explanation(p["section"], p["evidence_sentences"], p["score"])
        predictions[d["doc_id"]] = {"doc_id": d["doc_id"], "statute": preds}

    if save_predictions:
        path = os.path.join(cfg.out_dir, f"predictions_{run_name}.jsonl")
        with open(path, "w", encoding="utf-8") as f:
            for r in predictions.values():
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
        print("Saved:", path)

    gold = [d["gold_sections"] for d in test_split]
    pred = [[p["section"] for p in predictions[d["doc_id"]]["statute"]] for d in test_split]
    labels = sorted({l for ls in gold + pred for l in ls})
    mlb = MultiLabelBinarizer(classes=labels)
    yt, yp = mlb.fit_transform(gold), mlb.transform(pred)
    metrics = {
        "macro_f1": f1_score(yt, yp, average="macro", zero_division=0),
        "micro_f1": f1_score(yt, yp, average="micro", zero_division=0),
        "weighted_f1": f1_score(yt, yp, average="weighted", zero_division=0),
        "macro_precision": precision_score(yt, yp, average="macro", zero_division=0),
        "macro_recall": recall_score(yt, yp, average="macro", zero_division=0),
        "micro_precision": precision_score(yt, yp, average="micro", zero_division=0),
        "micro_recall": recall_score(yt, yp, average="micro", zero_division=0),
        "exact_match_accuracy": accuracy_score(yt, yp),
        "hamming_loss": hamming_loss(yt, yp),
    }
    print(f"\n--- {run_name}: TEST metrics ---")
    for k, v in metrics.items():
        print(f"{k:>22s}: {v:.4f}")
    report = pd.DataFrame(classification_report(yt, yp, target_names=labels, zero_division=0, output_dict=True)).T
    report.to_csv(os.path.join(cfg.out_dir, f"per_class_{run_name}.csv"))
    with open(os.path.join(cfg.out_dir, f"metrics_{run_name}.json"), "w") as f:
        json.dump(metrics, f, indent=2)
    return metrics, predictions


# Run 2: prototype pipeline with the ORIGINAL (not fine-tuned) InLegalBERT
run2_metrics, run2_predictions = evaluate_run("run2_prototype_baseline")

## 11. Run 1 – prototype-contrastive fine-tuning

Anchor = case sentence, positive = its applicable statute prototype, negatives = mined hard-negative
prototypes (plus the other in-batch prototypes). Hard negatives are mined with the original encoder,
then the encoder is fine-tuned with InfoNCE.

In [ ]:
# hard-negative prototypes mined with the original encoder
base_bank = build_prototype_bank(encoder)
hard_negative_map = mine_hard_negative_prototypes(base_bank, cfg.num_hard_negatives, cfg.exclude_same_base_section)
demo = next((c for c in ("IPC 302", "IPC 498A") if c in hard_negative_map), prototype_codes[0])
print(f"Hard-negative prototypes for {demo}: {hard_negative_map[demo]}")

train_pairs = []
for rec in train_split:
    train_pairs.extend(positive_pairs_from_explanation(rec["fact"], rec.get("explanation", {}) or {}))
print(f"{len(train_pairs)} (sentence, positive prototype) training pairs")


class SentencePrototypePairs(Dataset):
    def __init__(self, pairs): self.pairs = pairs
    def __len__(self): return len(self.pairs)
    def __getitem__(self, i): return self.pairs[i]

def collate_pairs(batch):
    return [s for s, _ in batch], [c for _, c in batch]

pair_loader = DataLoader(SentencePrototypePairs(train_pairs), batch_size=cfg.batch_size,
                         shuffle=True, collate_fn=collate_pairs)

In [ ]:
optimizer = torch.optim.AdamW([p for p in encoder.parameters() if p.requires_grad],
                              lr=cfg.encoder_lr, weight_decay=cfg.weight_decay)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

loss_history = []
for epoch in range(1, cfg.num_epochs + 1):
    encoder.train()
    running, steps = 0.0, 0
    optimizer.zero_grad()
    for step, (sentences, pos_codes) in enumerate(pair_loader, start=1):
        positives = list(dict.fromkeys(pos_codes))                       # unique positive prototypes in batch
        hard = []
        for c in positives:
            for n in hard_negative_map[c]:
                if n not in positives and n not in hard:
                    hard.append(n)
        batch_prototypes = positives + hard                              # positives + hard-negative prototypes
        targets = torch.tensor([batch_prototypes.index(c) for c in pos_codes], device=DEVICE)

        with torch.amp.autocast("cuda", enabled=USE_AMP):
            sent_emb = encoder(sentences, cfg.max_length)
            proto_emb = encoder([prototype_texts[c] for c in batch_prototypes], cfg.max_length)
            logits = sent_emb @ proto_emb.t() / cfg.temperature
            loss = F.cross_entropy(logits.float(), targets)              # InfoNCE over prototypes

        scaler.scale(loss / cfg.grad_accum_steps).backward()
        if step % cfg.grad_accum_steps == 0 or step == len(pair_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_([p for p in encoder.parameters() if p.requires_grad], cfg.grad_clip)
            scaler.step(optimizer); scaler.update(); optimizer.zero_grad()
        running += loss.item(); steps += 1

    loss_history.append(running / max(1, steps))
    print(f"[prototype-contrastive] epoch {epoch}/{cfg.num_epochs}  InfoNCE loss = {loss_history[-1]:.4f}")

torch.save(encoder.state_dict(), os.path.join(cfg.out_dir, "prototype_contrastive_encoder.pt"))

## 12. Run 1 evaluation (prototype-contrastive InLegalBERT) and Run 1 vs Run 2

In [ ]:
run1_metrics, run1_predictions = evaluate_run("run1_prototype_contrastive")

comparison = pd.DataFrame({
    "Run 1 - Prototype-Contrastive InLegalBERT": run1_metrics,
    "Run 2 - Prototype InLegalBERT (no fine-tuning)": run2_metrics,
}).loc[["macro_f1", "micro_f1", "exact_match_accuracy"]].round(4)
print(comparison.to_string())
comparison.to_csv(os.path.join(cfg.out_dir, "run1_vs_run2.csv"))

with open(os.path.join(cfg.out_dir, "comparison_pred_vs_gold.csv"), "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["doc_id", "gold", "run1_predicted", "run2_predicted", "run1_exact"])
    for d in test_split:
        g = set(d["gold_sections"])
        p1 = {p["section"] for p in run1_predictions[d["doc_id"]]["statute"]}
        p2 = {p["section"] for p in run2_predictions[d["doc_id"]]["statute"]}
        w.writerow([d["doc_id"], "; ".join(sorted(g)), "; ".join(sorted(p1)), "; ".join(sorted(p2)), "YES" if g == p1 else "NO"])
print("DONE.")